
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Guideline Judges with MLflow
## Overview
In this demo, we'll implement both global and per-row guideline judges. We'll define natural-language evaluation rules, run them against evaluation datasets, and inspect the results, including evaluating pre-generated outputs without calling the agent.

## Learning Objectives
By the end of this demo, you will be able to:
1. Implement the `Guidelines` judge for uniform evaluation criteria across all rows
2. Evaluate pre-generated inputs/outputs without a predict function
3. Apply `ExpectationsGuidelines` for per-row, scenario-specific evaluation

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Prerequisites</strong>
<p style="margin: 8px 0 0 0; color: #333;">This demo uses the agent created in <strong>Demo - Agent Setup</strong>. Please ensure you have completed that notebook before proceeding.</p>
</div>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup
Run the following cell to configure your environment.

In [0]:
%run ../Includes/Classroom-Setup-3

Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-ee2a45f6-c621-4b89-b1bb-f4bc2f7bbb77
Resolved 144 packages in 3.78s
 Downloaded hf-xet
 Downloaded kiwisolver
 Downloaded pydantic-core
 Downloaded tiktoken
 Downloaded zstandard
 Downloaded aiohttp
 Downloaded tokenizers
 Downloaded grpcio
 Downloaded databricks-sdk
 Downloaded cryptography
 Downloaded pillow
 Downloaded sqlalchemy
 Downloaded fonttools
 Downloaded mlflow-tracing
 Downloaded matplotlib
 Downloaded pyarrow
 Downloaded numpy
 Downloaded scikit-learn
 Downloaded mlflow
 Downloaded pandas
 Downloaded scipy
 Downloaded openai
 Downloaded litellm
 Downloaded botocore
Prepared 130 packages in 7.17s
Uninstalled 106 packages in 62ms
Installed 130 packages in 1.45s
 - aiohappyeyeballs==2.4.4
 + aiohappyeyeballs==2.7.1
 - aiohttp==3.11.10
 + aiohttp==3.14.5
 - aiosignal==1.2.0
 + aiosignal==1.4.0
 + alembic==1.20.0
 - annotated-doc==0.0.4
 + annotated-doc==0.0.5
 - annotated-types==0.7.0
 + annotated-

Pre-installed Databricks Runtime package(s) replaced: frozenlist: 1.5.0 -> 1.8.0, multidict: 6.1.0 -> 7.1.0, regex: 2024.11.6 -> 2026.9.29, packaging: 24.2 -> 25.0, jsonpointer: 3.0.0 -> 3.3, gitdb: 4.0.11 -> 4.0.12, MarkupSafe: 3.0.2 -> 3.0.4, importlib_metadata: 8.5.0 -> 8.9.0, starlette: 0.50.0 -> 1.9.0, huggingface_hub: 1.2.4 -> 2.2.0, jsonpatch: 1.33 -> 1.35, langsmith: 0.6.1 -> 0.14.11, jmespath: 1.0.1 -> 1.1.0, google-crc32c: 1.8.0 -> 1.9.0, uuid_utils: 0.12.0 -> 0.17.1, google-cloud-core: 2.5.0 -> 2.8.0, orjson: 3.11.5 -> 3.13.0, matplotlib: 3.10.0 -> 3.11.2, tokenizers: 0.22.2 -> 0.23.3, cycler: 0.11.0 -> 0.12.1, idna: 3.7 -> 3.20, cryptography: 44.0.1 -> 46.0.7, grpcio: 1.67.0 -> 1.67.1, scikit-learn: 1.6.1 -> 1.9.1, cachetools: 5.5.1 -> 6.2.6, pytz: 2024.1 -> 2026.5, mypy-extensions: 1.0.0 -> 1.1.0, tzdata: 2024.1 -> 2026.5, langchain-core: 1.2.6 -> 1.6.12, googleapis-common-protos: 1.65.0 -> 1.75.5, annotated-types: 0.7.0 -> 0.8.0, opentelemetry-sdk: 1.39.1 -> 1.45.1, scipy

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-ee2a45f6-c621-4b89-b1bb-f4bc2f7bbb77
Resolved 1 package in 116ms
Prepared 1 package in 86ms
Installed 1 package in 1ms
 + backoff==2.2.1
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-ee2a45f6-c621-4b89-b1bb-f4bc2f7bbb77
Resolved 129 packages in 1.94s
 Downloaded openai-agents
 Downloaded mlflow-tracing
 Downloaded databricks-connect
 Downloaded mlflow-skinny
 Downloaded mlflow
Prepared 28 packages in 808ms
Uninstalled 11 packages in 33ms
Installed 29 packages in 210ms
 + aiohttp-retry==2.9.1
 - blinker==1.7.0
 ~ blinker==1.9.0
 + databricks-ai-bridge==0.22.0
 + databricks-ai-search==0.79
 - databricks-connect==18.1.9
 + databricks-con

Pre-installed Databricks Runtime package(s) replaced: googleapis-common-protos: 1.65.0 -> 1.75.0, protobuf: 5.29.4 -> 5.29.6, databricks-connect: 18.1.9 -> 17.0.10, mlflow-skinny: 3.8.1 -> 3.17.1, PyJWT: 2.10.1 -> 2.15.1, wcwidth: 0.2.5 -> 0.9.2. uv pip install may report the previous version as uninstalled, but the runtime's preinstalled packages remain on disk.


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-ee2a45f6-c621-4b89-b1bb-f4bc2f7bbb77
Resolved 43 packages in 524ms
Prepared 1 package in 125ms
Uninstalled 1 package in 0.66ms
Installed 1 package in 2ms
 - databricks-vectorsearch==0.75
 + databricks-vectorsearch==0.68
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


...

Your username is: labuser17704131_1792356850@vocareum.com
The location of your experiment: /Users/labuser17704131_1792356850@vocareum.com/airbnb_experiment


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


labuser17704131_1792356850.demo_agent_eval.airbnb


2026/10/18 23:05:57 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - mlflow (current: 3.17.1, required: mlflow==3.13.0)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-ee2a45f6-c621-4b89-b1bb-f4bc2f7bbb77/lib/python3.12/site-packages/databricks/connect/session.py:476: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


Guidelines Endpoint: databricks:/databricks-gpt-oss-120b


In [0]:
print(f"Username:          {username}")
print(f"Catalog Name:      {catalog_name}")
print(f"Schema Name:       {schema_name}")

Username:          labuser17704131_1792356850@vocareum.com
Catalog Name:      labuser17704131_1792356850
Schema Name:       demo_agent_eval


## B. Load Evaluation Datasets

We'll use three datasets stored in the UC volume, each demonstrating a different guideline evaluation pattern:
- **`guidelines_eval.json`** - Inputs only (agent generates outputs at evaluation time)
- **`guidelines_eval_pre_gen.json`** - Pre-generated inputs and outputs (no agent call needed)
- **`guidelines_eval_row_level.json`** - Includes per-row guidelines in the `expectations` field

In [0]:
import json
from pprint import pprint
from pathlib import Path

def read_eval_from_vol(file_name: str):
    path = Path(f"/Volumes/{catalog_name}/{schema_name}/agent_vol/{file_name}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

guidelines_dataset = read_eval_from_vol("guidelines_eval.json")
guidelines_dataset_pre_gen = read_eval_from_vol("guidelines_eval_pre_gen.json")
guidelines_dataset_row_level = read_eval_from_vol("guidelines_eval_row_level.json")


### B1. Inspect Dataset Structure

Let's examine the first dataset to understand the expected schema. Each entry contains an `inputs` field with the user's request: this is what gets passed to the agent during evaluation.

In [0]:
pprint(guidelines_dataset)

[{'inputs': {'input': [{'content': 'Can you tell me what the average price is '
                                   'in Mission?',
                        'role': 'user'}]}},
 {'inputs': {'input': [{'content': 'How many rooms in Mission have a private '
                                   'room?',
                        'role': 'user'}]}}]


## C. Global Guidelines Evaluation

The `Guidelines` judge applies the same rules to every row. Here we create a guideline requiring responses in Spanish. Both inputs will fail since the agent responds in English.

In [0]:
from mlflow.genai.scorers import Guidelines

language_guideline = Guidelines(
    name="spanish",
    guidelines=["The response should be in Spanish"],
    model_name=guidelines_endpoint
)

### C1. Run Evaluation with Agent

In [0]:
guidelines_dataset_results = mlflow.genai.evaluate(
    data=guidelines_dataset,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[language_guideline]
)

2026/10/18 23:08:35 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/10/18 23:08:35 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/2 [Elapsed: 00:00, Remaining: ?]

### C2. Inspect Results

In [0]:
print(f"The run ID is: {guidelines_dataset_results.run_id}")
print(f"The aggregated metrics are: {guidelines_dataset_results.metrics}")
print("\nThe results from the previous batch of inputs:")
display(guidelines_dataset_results.result_df)

The run ID is: c8567403c3294dac9e20a72e5c6cd130
The aggregated metrics are: {'spanish/mean': np.float64(0.0)}

The results from the previous batch of inputs:


trace_id spanish/value spanish/rationale trace client_request_id state request_time execution_duration request response trace_metadata tags spans assessments tr-cf129ae82ae8ee3aea5cd99c6d6faabe no The guideline states that the response should be in Spanish. However, the provided response is in English. Therefore, the guideline is not satisfied because the response does not meet the language requirement specified in the guideline. {"info": {"trace_id": "tr-cf129ae82ae8ee3aea5cd99c6d6faabe", "client_request_id": "tr-cf129ae82ae8ee3aea5cd99c6d6faabe", "trace_location": {"type": "MLFLOW_EXPERIMENT", "mlflow_experiment": {"experiment_id": "4107611117782955"}}, "request_time": "2026-10-18T23:08:47.377Z", "state": "OK", "trace_metadata": {"mlflow.sourceRun": "c8567403c3294dac9e20a72e5c6cd130", "mlflow.trace_schema.version": "3", "mlflow.databricks.workspaceID": "7474659189051797", "mlflow.trace.tokenUsage": "{\"input_tokens\": 701, \"output_tokens\": 134, \"total_tokens\": 835, \"cache_read_input_tokens\": 576}", "mlflow.modelId": "m-76cbb6e9978c4d179ff19d82411101ef", "mlflow.databricks.notebookID": "4107611117782939", "mlflow.databricks.notebookPath": "/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.4 Demo - Guideline Judges with MLflow", "mlflow.databricks.workspaceURL": "https://dbc-4d4ce156-516a.cloud.databricks.com", "mlflow.source.name": "/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.4 Demo - Guideline Judges with MLflow", "mlflow.traceOutputs": "{\"tool_choice\": null, \"truncation\": null, \"id\": null, \"created_at\": null, \"error\": null, \"incomplete_details\": null, \"instructions\": null, \"metadata\": null, \"model\": null, \"object\": \"response\", \"output\": [{\"type\": \"function_call\", \"id\": \"chatcmpl_...", "mlflow.traceInputs": "{\"request\": {\"tool_choice\": null, \"truncation\": null, \"max_output_tokens\": null, \"metadata\": null, \"parallel_tool_calls\": null, \"tools\": null, \"reasoning\": null, \"store\": null, \"stream\": null, \"temperature\": null, \"text\": null, \"top_p\": null, \"use...", "mlflow.trace.sizeStats": "{\"total_size_bytes\": 29620, \"num_spans\": 5, \"max\": 10129, \"p25\": 2285, \"p50\": 4160, \"p75\": 10005}", "mlflow.source.type": "NOTEBOOK", "mlflow.databricks.notebook.commandID": "1001792364389784_8891753721484109853_e80992c6a309421b9f5df667d711205b", "mlflow.user": "spark-ee2a45f6-c621-4b89-b1bb-f4", "mlflow.trace.sizeBytes": "29620", "mlflow.databricks.webappURL": "https://dbc-4d4ce156-516a.cloud.databricks.com"}, "assessments": [{"assessment_id": "a-e6de6454c7ef452993725180bcf4ad00", "assessment_name": "spanish", "trace_id": "tr-cf129ae82ae8ee3aea5cd99c6d6faabe", "span_id": "c48e71dc951b634c", "source": {"source_type": "LLM_JUDGE", "source_id": "databricks"}, "create_time": "2026-10-18T23:08:55.412Z", "last_update_time": "2026-10-18T23:08:55.412Z", "feedback": {"value": "no"}, "rationale": "The guideline states that the response should be in Spanish. However, the provided response is in English. Therefore, the guideline is not satisfied because the response does not meet the language requirement specified in the guideline.", "metadata": {"mlflow.assessment.sourceRunId": "c8567403c3294dac9e20a72e5c6cd130", "guideline": "The response should be in Spanish", "_user_defined_assessment_name": "true"}, "valid": true}], "tags": {"mlflow.artifactLocation": "dbfs:/databricks/mlflow-tracking/4107611117782955/tr-cf129ae82ae8ee3aea5cd99c6d6faabe/artifacts", "mlflow.user": "labuser17704131_1792356850@vocareum.com", "mlflow.eval.requestId": "48d8d396-d336-4372-9fba-bc06cbab68aa", "mlflow.traceName": "predict"}, "request_preview": "Can you tell me what the average price is in Mission?", "response_preview": "The average price of Airbnb listings in t

### C3. Evaluate Pre-Generated Outputs

When you already have inputs and outputs stored in a dataset, you can skip the `predict_fn` parameter entirely. The dataset `guidelines_eval_pre_gen.json` contains one correct Spanish response and one English response. Expect one pass and one fail.

In [0]:
guidelines_dataset_pre_gen_results = mlflow.genai.evaluate(
    data=guidelines_dataset_pre_gen,
    scorers=[language_guideline]
)

Evaluating:   0%|          | 0/2 [Elapsed: 00:00, Remaining: ?]


### C4. Inspect Pre-Generated Results

Since the dataset already contained outputs, the evaluation skipped agent invocation entirely. We expect one row to pass (Spanish response) and one to fail (English response).

In [0]:
print(f"The run ID is: {guidelines_dataset_pre_gen_results.run_id}")
print(f"The aggregated metrics are: {guidelines_dataset_pre_gen_results.metrics}")
print("\nThe results from the previous batch of inputs:")
display(guidelines_dataset_pre_gen_results.result_df)

The run ID is: a24bf8fdbfd24552a38555f757bae670
The aggregated metrics are: {'spanish/mean': np.float64(0.5)}

The results from the previous batch of inputs:


trace_id,spanish/value,spanish/rationale,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
tr-16f71216a11dbe6701664d852ba55b2e,yes,"The guideline states that the response should be in Spanish. The provided response is indeed in Spanish, as it gives the average price in the Mission neighborhood in Spanish. Therefore, the input fully complies with the guideline.","{""info"": {""trace_id"": ""tr-16f71216a11dbe6701664d852ba55b2e"", ""client_request_id"": ""tr-16f71216a11dbe6701664d852ba55b2e"", ""trace_location"": {""type"": ""MLFLOW_EXPERIMENT"", ""mlflow_experiment"": {""experiment_id"": ""4107611117782955""}}, ""request_time"": ""2026-10-18T23:10:38.174Z"", ""state"": ""OK"", ""trace_metadata"": {""mlflow.sourceRun"": ""a24bf8fdbfd24552a38555f757bae670"", ""mlflow.trace_schema.version"": ""3"", ""mlflow.databricks.workspaceID"": ""7474659189051797"", ""mlflow.databricks.notebookPath"": ""/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.4 Demo - Guideline Judges with MLflow"", ""mlflow.databricks.notebookID"": ""4107611117782939"", ""mlflow.databricks.workspaceURL"": ""https://dbc-4d4ce156-516a.cloud.databricks.com"", ""mlflow.source.name"": ""/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.4 Demo - Guideline Judges with MLflow"", ""mlflow.traceOutputs"": ""{\""output\"": [\""El precio promedio de los alojamientos en el vecindario Mission es de $188.63 por noche.\""]}"", ""mlflow.traceInputs"": ""{\""input\"": [{\""role\"": \""user\"", \""content\"": \""Can you tell me what the average price is in Mission?\""}]}"", ""mlflow.trace.sizeStats"": ""{\""total_size_bytes\"": 2641, \""num_spans\"": 1, \""max\"": 691, \""p25\"": 691, \""p50\"": 691, \""p75\"": 691}"", ""mlflow.source.type"": ""NOTEBOOK"", ""mlflow.databricks.notebook.commandID"": ""1001792364389784_8891753721484109853_e80992c6a309421b9f5df667d711205b"", ""mlflow.user"": ""spark-ee2a45f6-c621-4b89-b1bb-f4"", ""mlflow.trace.sizeBytes"": ""2641"", ""mlflow.databricks.webappURL"": ""https://dbc-4d4ce156-516a.cloud.databricks.com""}, ""assessments"": [{""assessment_id"": ""a-1699d38bda11431c878b8bb3d0dfce2f"", ""assessment_name"": ""spanish"", ""trace_id"": ""tr-16f71216a11dbe6701664d852ba55b2e"", ""span_id"": ""8e756e0a43ef4d21"", ""source"": {""source_type"": ""LLM_JUDGE"", ""source_id"": ""databricks""}, ""create_time"": ""2026-10-18T23:10:40.058Z"", ""last_update_time"": ""2026-10-18T23:10:40.058Z"", ""feedback"": {""value"": ""yes""}, ""rationale"": ""The guideline states that the response should be in Spanish. The provided response is indeed in Spanish, as it gives the average price in the Mission neighborhood in Spanish. Therefore, the input fully complies with the guideline."", ""metadata"": {""mlflow.assessment.sourceRunId"": ""a24bf8fdbfd24552a38555f757bae670"", ""guideline"": ""The response should be in Spanish"", ""_user_defined_assessment_name"": ""true""}, ""valid"": true}], ""tags"": {""mlflow.artifactLocation"": ""dbfs:/databricks/mlflow-tracking/4107611117782955/tr-16f71216a11dbe6701664d852ba55b2e/artifacts"", ""mlflow.user"": ""labuser17704131_1792356850@vocareum.com"", ""mlflow.eval.requestId"": ""527c5c9b4fd77e98d6e2c6d49fb3aed91af64c29ab95324a7afad8d326c49e0c"", ""mlflow.traceName"": ""root_span""}, ""request_preview"": ""Can you tell me what the average price is in Mission?"", ""response_preview"": ""{\""output\"": [\""El precio promedio de los alojamientos en el vecindario Mission es de $188.63 por noche.\""]}"", ""execution_duration_ms"": 0}, ""data"": {""spans"": [{""trace_id"": ""FvcSFqEdvmcBZk2FK6VbLg=="", ""span_id"": ""jnVuCkPvTSE="", ""parent_span_id"": null, ""name"": ""root_span"", ""start_time_unix_nano"": 1792365038174865527, ""end_time_

## D. Per-Row Guidelines with ExpectationsGuidelines

`ExpectationsGuidelines` applies different guidelines to each row using the `expectations` field in the dataset. This is useful when different interactions require different evaluation criteria.

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;"><code>ExpectationsGuidelines</code> requires an <code>outputs</code> field.</strong>
<p style="margin: 8px 0 0 0; color: #333;">They can be passed directly in the dataset or as a trace containing them. We will pass them directly.</p>
</div>
</div>
</div>


### D1. Define and Run Per-Row Evaluation

Unlike global `Guidelines`, `ExpectationsGuidelines` reads the `expectations` field from each row in the dataset. Each row can specify its own list of guidelines, enabling scenario-specific evaluation without separate evaluation runs.

In [0]:
from mlflow.genai.scorers import ExpectationsGuidelines

expected_guidelines = ExpectationsGuidelines(
    name="expected_guidelines",
    model_name=guidelines_endpoint
)

guidelines_dataset_row_level_results = mlflow.genai.evaluate(
    data=guidelines_dataset_row_level,
    scorers=[expected_guidelines]
)

Evaluating:   0%|          | 0/2 [Elapsed: 00:00, Remaining: ?]


### D2. Inspect Per-Row Results

Each row is scored against its own `expectations` guidelines. Rows meeting their specific criteria pass; those that don't will show which guideline was violated.

In [0]:
print(f"The run ID is: {guidelines_dataset_row_level_results.run_id}")
print(f"The aggregated metrics are: {guidelines_dataset_row_level_results.metrics}")
print("\nThe results from the previous batch of inputs:")
display(guidelines_dataset_row_level_results.result_df)

The run ID is: e87ed7cc6ea1466db28ca85452ce0605
The aggregated metrics are: {'expected_guidelines/mean': np.float64(0.5)}

The results from the previous batch of inputs:


trace_id expected_guidelines/value expected_guidelines/rationale guidelines/value trace client_request_id state request_time execution_duration request response trace_metadata tags spans assessments tr-215e7a725aa414ce209132548ba6a593 no The guideline states that the response should include the word 'mean'. In the provided response, the output does not contain the word 'mean' at all. Therefore, the guideline is not satisfied. The response only mentions the expected price without addressing the average in terms of 'mean'. null {"info": {"trace_id": "tr-215e7a725aa414ce209132548ba6a593", "client_request_id": "tr-215e7a725aa414ce209132548ba6a593", "trace_location": {"type": "MLFLOW_EXPERIMENT", "mlflow_experiment": {"experiment_id": "4107611117782955"}}, "request_time": "2026-10-18T23:19:14.368Z", "state": "OK", "trace_metadata": {"mlflow.sourceRun": "e87ed7cc6ea1466db28ca85452ce0605", "mlflow.trace_schema.version": "3", "mlflow.databricks.workspaceID": "7474659189051797", "mlflow.databricks.notebookPath": "/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.4 Demo - Guideline Judges with MLflow", "mlflow.databricks.notebookID": "4107611117782939", "mlflow.databricks.workspaceURL": "https://dbc-4d4ce156-516a.cloud.databricks.com", "mlflow.source.name": "/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.4 Demo - Guideline Judges with MLflow", "mlflow.traceOutputs": "{\"output\": [\"The expected price for listings in this neighborhood is $188.63 per night.\"]}", "mlflow.traceInputs": "{\"input\": [{\"role\": \"user\", \"content\": \"Can you tell me what the average price is in Mission?\"}]}", "mlflow.trace.sizeStats": "{\"total_size_bytes\": 2599, \"num_spans\": 1, \"max\": 677, \"p25\": 677, \"p50\": 677, \"p75\": 677}", "mlflow.source.type": "NOTEBOOK", "mlflow.databricks.notebook.commandID": "1001792364389784_8891753721484109853_e80992c6a309421b9f5df667d711205b", "mlflow.user": "spark-ee2a45f6-c621-4b89-b1bb-f4", "mlflow.trace.sizeBytes": "2599", "mlflow.databricks.webappURL": "https://dbc-4d4ce156-516a.cloud.databricks.com"}, "assessments": [{"assessment_id": "a-d38c851f3d0149fe8c944f652076b0dd", "assessment_name": "expected_guidelines", "trace_id": "tr-215e7a725aa414ce209132548ba6a593", "span_id": "78ccaf1e213e98c7", "source": {"source_type": "LLM_JUDGE", "source_id": "databricks"}, "create_time": "2026-10-18T23:19:16.950Z", "last_update_time": "2026-10-18T23:19:16.950Z", "feedback": {"value": "no"}, "rationale": "The guideline states that the response should include the word 'mean'. In the provided response, the output does not contain the word 'mean' at all. Therefore, the guideline is not satisfied. The response only mentions the expected price without addressing the average in terms of 'mean'.", "metadata": {"mlflow.assessment.sourceRunId": "e87ed7cc6ea1466db28ca85452ce0605", "guideline": "The response should have the word 'mean' in its answer.", "_user_defined_assessment_name": "true"}, "valid": true}, {"assessment_id": "a-9211c9096dbf48e9bbe707de25936bfa", "assessment_name": "guidelines", "trace_id": "tr-215e7a725aa414ce209132548ba6a593", "span_id": "78ccaf1e213e98c7", "source": {"source_type": "HUMAN", "source_id": "spark-ee2a45f6-c621-4b89-b1bb-f4"}, "create_time": "2026-10-18T23:19:17.033Z", "last_update_time": "2026-10-18T23:19:17.033Z", "expectation": {"serialized_value": {"serialization_format": "JSON_FORMAT", "value": "[\"The response should have the word 'mean' in its answer.\"]"}}, "metadata": {"mlflow.assessment.sourceRunId": "e87ed7cc6ea1466db28ca85452ce0605"}}], "tags": {"mlflow.artifactLocation": "dbfs:/databricks/mlflow-tracking/4107611117782955/tr-215e7a725aa414ce209132548ba6a593/artifacts", "mlflow.user": "labuser17704131_1792356850@vocareum.com", "mlflow.eval.requestId":

## E. Conclusion
In this demo, you:

1. Applied global `Guidelines` to evaluate all rows against uniform criteria (Spanish language requirement)
2. Evaluated pre-generated outputs without calling the agent by omitting `predict_fn`
3. Used `ExpectationsGuidelines` for per-row, scenario-specific evaluation criteria

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>

